# Data Exploration

The code was partially adapted from the following: <br>
https://neuro.inf.unibe.ch/AlgorithmsNeuroscience/Tutorial_files/DataVisualization.html 
https://www.cs.colostate.edu/eeg/data/json/doc/tutorial/_build/html/p300_single_subject.html 
https://github.com/Mentalab-hub/explorepy/blob/master/examples/p300_demo/analysis_csv_4_channel.py 

Loading the data + inspecting the column entry: 

In [ ]:
from project_config import EEG_DATA_DIR
import pandas as pd
header_file_path = str(EEG_DATA_DIR / 'Header.csv')
header_df = pd.read_csv(header_file_path)
header_df.head()

In [ ]:
# usingn one session to explore the data
session_file_path = str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv')
session_df = pd.read_csv(session_file_path, header=None, names=header_df.columns)
session_df.head()

Plotting the raw EEG signals:

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 10))
plt.suptitle("non-filtered EEGs (1-24 Hz)") 
channels_to_plot = ["FP1", "F3", "Cz", "Pz", "O1"] # chosen for frontal, central, parietal, occipital
time_window = (0, 1)  # seconds

for i, channel in enumerate(channels_to_plot, 1):
    plt.subplot(len(channels_to_plot), 1, i)
    plt.plot(session_df["Time"], session_df[channel])
    plt.title(f"EEG signal of {channel}")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")

plt.tight_layout()
plt.show()

Calculate the differences between consecutive time points in the 'Time' column using np.diff. If the 'Time' column is in seconds as I thought, the average difference should be consistent with the sampling frequency.

In [ ]:
import pandas as pd
import numpy as np

time_diffs = np.diff(session_df['Time'])
print("avg time difference between samples:", np.mean(time_diffs), "seconds") # check that time is in s 
trigger_indices = session_df[session_df["Trigger"] == 1].index # verify trigger --> extracy indices where its 1
trigger_times_seconds = session_df.loc[trigger_indices, "Time"] # indices to timestamp
print("example trigger times in s:", trigger_times_seconds.head())
print("range of trigger times in s: ", trigger_times_seconds.min(), ",", trigger_times_seconds.max())


# Preprocessing:

This preprocessing was mostly adapted from the researcher who created the dataset's own preprocessing code in their GitHub repository: https://github.com/plcrodrigues/py.BI.EEG.2015a-GIPSA. <br>
The parameters used here are the same as the ones utilized by the researchers.

In [ ]:
import numpy as np 
fs = 512  # in hz
samples_per_epoch = int(0.8 * fs) 
trigger_indices = np.where(session_df["Trigger"] == 1)[0] # trigger points which is onset of an epoch
epochs = [] # 3d
labels = []
for index in trigger_indices:# extraction of epochs
    if (index + samples_per_epoch) <= len(session_df):  #check if there is data after trigger point for one full epoch
        epoch = session_df.iloc[index:index+samples_per_epoch, 1:-2].values  #exclude Time, Trigger, and Target columns
        label = session_df.iloc[index]["Target"]
        epochs.append(epoch)
        labels.append(label)
epochs = np.array(epochs)
labels = np.array(labels)
print(f"num of epochs: {len(epochs)}")
print(f"shapes of epochs: {epochs.shape}")

Plotting filtered data:
- the filter function was adapted from the one used here: https://www.tutorialspoint.com/digital-band-pass-butterworth-filter-in-python): <br>
- the plotting code was adapted from the one shown here: https://plotly.com/python/v3/ipython-notebooks/mne-tutorial/

In [ ]:
import numpy as np
from scipy.signal import butter, lfilter, filtfilt

def bandpass_filter(data, lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype="band")
    y = filtfilt(b, a, data)  #  filtfilt for zero-phase filtering
    return y

'''
# normalizing for very large or very small values
for column in header_df.columns[1:-2]: 
    session_df[column] -= session_df[column].mean()
    session_df[column] /= session_df[column].std()
'''
for channel in header_df.columns[1:-2]: 
    session_df[channel] = bandpass_filter(session_df[channel], 1, 24, fs)
plt.figure(figsize=(15, 10))
plt.suptitle("filtered EEG signals (1-24 Hz)")
for i, channel in enumerate(channels_to_plot, 1):
    plt.subplot(len(channels_to_plot), 1, i)
    # plot data segment to avoid spike
    plt.plot(session_df["Time"][1:], session_df[channel][1:])
    plt.title(f"EEG signal for {channel}")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (uV)")
plt.tight_layout()
plt.show()


This tutorial was also used to understand how to deal with baselining and its related parameters: https://eegget-it.nl/erp_analysis.html


Averaging across all trials of the single session for a single participant (the filtering and epoching applied to the classifiers is done through the MNE library and is shown in the classifiers' notebook):

In [ ]:
# epoch extraction
t_before = 0.2  # 200 ms before the trigger
t_after = 0.8   # 800 ms after the trigger
fs = 512        # s freq
n_samples = int((t_before + t_after) * fs)
# epoch + avg
trigger_times = session_df[session_df["Trigger"] == 1]["Time"]
epochs = {channel: [] for channel in channels_to_plot}
for trigger_time in trigger_times:
    epoch_window = (session_df["Time"] >= trigger_time - t_before) & (session_df["Time"] < trigger_time + t_after) # here im creating the epochs based on where the trigger tiems are
    for channel in channels_to_plot: # for each channel
        epochs[channel].append(session_df[epoch_window][channel].values[:n_samples])
average_trials = {channel: np.mean(np.vstack(epochs[channel]), axis=0) for channel in channels_to_plot}
plt.figure(figsize=(15, 10))
for i, channel in enumerate(channels_to_plot, 1):
    plt.subplot(len(channels_to_plot), 1, i)
    plt.plot(np.linspace(-t_before, t_after, n_samples), average_trials[channel])
    plt.title(f"average EEG signal for {channel}")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (uV)")
plt.tight_layout()
plt.show()
# epoching based on target vs non target condition now
target_epochs = {channel: [] for channel in channels_to_plot}
non_target_epochs = {channel: [] for channel in channels_to_plot}
for trigger_time, target in zip(trigger_times, session_df[session_df["Trigger"] == 1]["Target"]):
    epoch_window = (session_df["Time"] >= trigger_time - t_before) & (session_df["Time"] < trigger_time + t_after) # calculaitng the start and end of the epoch based on the trigger
    for channel in channels_to_plot:
        epoch = session_df[epoch_window][channel].values[:n_samples]
        if target == 1:
            target_epochs[channel].append(epoch)
        else:
            non_target_epochs[channel].append(epoch)
erp_target = {channel: np.mean(np.vstack(target_epochs[channel]), axis=0) for channel in channels_to_plot}
erp_non_target = {channel: np.mean(np.vstack(non_target_epochs[channel]), axis=0) for channel in channels_to_plot}

'''
erp_target = {channel: np.mean(np.vstack(target_epochs[channel]), axis=1) for channel in channels_to_plot}
erp_non_target = {channel: np.mean(np.vstack(non_target_epochs[channel]), axis=1) for channel in channels_to_plot}
'''

plt.figure(figsize=(15, 10))
for i, channel in enumerate(channels_to_plot, 1):
    plt.subplot(len(channels_to_plot), 1, i)
    plt.plot(np.linspace(-t_before, t_after, n_samples), erp_target[channel], label="Target")
    plt.plot(np.linspace(-t_before, t_after, n_samples), erp_non_target[channel], label="Non-Target")
    plt.title(f"Target vs Non-Target ERP for {channel}")
    plt.xlabel("time (s)")
    plt.ylabel("amplitude (uV)")
    plt.gca().invert_yaxis() 
    plt.legend()
plt.tight_layout()
plt.show()

Checking for unique values:

In [ ]:
unique_trigger_values = session_df.iloc[:, -2].unique()
print("unique Trigger values:", unique_trigger_values)
unique_target_values = session_df.iloc[:, -1].unique()
print("unique Target values:", unique_target_values)

Plot over all participants:

In [ ]:
header_df = pd.read_csv(header_file_path)
csv_file_paths = [ str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_05_csv/subject_05_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_05_csv/subject_05_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_05_csv/subject_05_session_03.csv'), 
    str(EEG_DATA_DIR / 'subject_06_csv/subject_06_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_06_csv/subject_06_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_06_csv/subject_06_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_07_csv/subject_07_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_07_csv/subject_07_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_07_csv/subject_07_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_08_csv/subject_08_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_08_csv/subject_08_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_08_csv/subject_08_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_09_csv/subject_09_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_09_csv/subject_09_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_09_csv/subject_09_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_10_csv/subject_10_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_10_csv/subject_10_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_10_csv/subject_10_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_11_csv/subject_11_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_11_csv/subject_11_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_11_csv/subject_11_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_12_csv/subject_12_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_12_csv/subject_12_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_12_csv/subject_12_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_13_csv/subject_13_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_13_csv/subject_13_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_13_csv/subject_13_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_14_csv/subject_14_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_14_csv/subject_14_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_14_csv/subject_14_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_15_csv/subject_15_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_15_csv/subject_15_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_15_csv/subject_15_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_16_csv/subject_16_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_16_csv/subject_16_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_16_csv/subject_16_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_17_csv/subject_17_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_17_csv/subject_17_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_17_csv/subject_17_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_18_csv/subject_18_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_18_csv/subject_18_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_18_csv/subject_18_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_19_csv/subject_19_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_19_csv/subject_19_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_19_csv/subject_19_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_20_csv/subject_20_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_20_csv/subject_20_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_20_csv/subject_20_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_21_csv/subject_21_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_21_csv/subject_21_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_21_csv/subject_21_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_22_csv/subject_22_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_22_csv/subject_22_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_22_csv/subject_22_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_23_csv/subject_23_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_23_csv/subject_23_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_23_csv/subject_23_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_24_csv/subject_24_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_24_csv/subject_24_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_24_csv/subject_24_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_25_csv/subject_25_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_25_csv/subject_25_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_25_csv/subject_25_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_26_csv/subject_26_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_26_csv/subject_26_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_26_csv/subject_26_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_28_csv/subject_28_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_28_csv/subject_28_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_28_csv/subject_28_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_29_csv/subject_29_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_29_csv/subject_29_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_29_csv/subject_29_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_30_csv/subject_30_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_30_csv/subject_30_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_30_csv/subject_30_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_31_csv/subject_31_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_31_csv/subject_31_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_31_csv/subject_31_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_32_csv/subject_32_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_32_csv/subject_32_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_32_csv/subject_32_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_33_csv/subject_33_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_33_csv/subject_33_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_33_csv/subject_33_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_34_csv/subject_34_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_34_csv/subject_34_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_34_csv/subject_34_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_35_csv/subject_35_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_35_csv/subject_35_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_35_csv/subject_35_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_36_csv/subject_36_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_36_csv/subject_36_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_36_csv/subject_36_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_37_csv/subject_37_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_37_csv/subject_37_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_37_csv/subject_37_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_38_csv/subject_38_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_38_csv/subject_38_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_38_csv/subject_38_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_39_csv/subject_39_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_39_csv/subject_39_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_39_csv/subject_39_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_40_csv/subject_40_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_40_csv/subject_40_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_40_csv/subject_40_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_41_csv/subject_41_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_41_csv/subject_41_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_41_csv/subject_41_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_42_csv/subject_42_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_42_csv/subject_42_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_42_csv/subject_42_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_43_csv/subject_43_session_01.csv'),
    str(EEG_DATA_DIR / 'subject_43_csv/subject_43_session_02.csv'),
    str(EEG_DATA_DIR / 'subject_43_csv/subject_43_session_03.csv'),
    ]
all_target_epochs = []
all_nontarget_epochs = []
for file_path in csv_file_paths:
    session_df = pd.read_csv(file_path, header=None, names=header_df.columns)
    for column in session_df.columns[1:-2]: 
        session_df[column] = bandpass_filter(session_df[column], 1, 24, fs, order=5) # apply filter 
    '''
        session_df[column] -= session_df[column].mean()
        session_df[column] /= session_df[column].std()
    '''
    trigger_times = session_df[session_df["Trigger"] == 1].index # extracting epochs
    for trigger_index in trigger_times:
        start = int(trigger_index - t_before * fs)
        end = int(trigger_index + t_after * fs)
        if start >= 0 and end < len(session_df):
            epoch = session_df.iloc[start:end, 1:-2].values
            if session_df.iloc[trigger_index]["Target"] == 1:
                all_target_epochs.append(epoch)
            else:
                all_nontarget_epochs.append(epoch)
all_target_epochs = np.array(all_target_epochs)
all_nontarget_epochs = np.array(all_nontarget_epochs)
avg_target_erp = np.mean(all_target_epochs, axis=(0, 2))
avg_nontarget_erp = np.mean(all_nontarget_epochs, axis=(0, 2))
time_vector = np.linspace(-t_before, t_after, n_samples)
plt.figure(figsize=(10, 6))
plt.plot(time_vector, avg_target_erp, label="Target", color="blue", linewidth=2)
plt.plot(time_vector, avg_nontarget_erp, label="Non-Target", color="red", linewidth=2)
plt.axvline(x=0, color="black", linestyle="--")
plt.axvspan(0.2, 0.5, color="green", alpha=0.3, label="expected P300 window") 
plt.title("averaged and smoothed ERP across all participants")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (uV)")
plt.gca().invert_yaxis() 
plt.legend()
plt.show()

### Visualizing class imbalance

Exploring 'Target' imbalance:

In [ ]:
header_path = str(EEG_DATA_DIR / 'Header.csv')
session_data_path = str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv')
header_df = pd.read_csv(header_path)
column_names = header_df.columns.tolist() # get col names
session_df = pd.read_csv(session_data_path, names=column_names)
target_counts = session_df["Target"].value_counts(normalize=True)
target_counts

In [ ]:
plt.figure(figsize=(10, 6))
target_counts.plot(kind="bar")
plt.title("distribution of Target column")
plt.xlabel("target value")
plt.ylabel("frequency")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.show()